# Monte Carlo Simulation Analysis

In [2]:
import numpy as np
import pandas as pd
import awkward as ak
import uproot

In [3]:
import matplotlib.pyplot as plt
from IPython.display import Image, display
from matplotlib.colors import LogNorm
# Aesthetics:
fs = 14    # fontsize

In [4]:
# Check out the structure
path = "/home/pira/Documenti/PoD/LCP/LCP_B/ALICE/AO2DtreeMC.root"
file = uproot.open(path)
file.classnames()

{'DF_2303121152302944;1': 'TDirectory',
 'DF_2303121152302944/O2mccollision;1': 'TTree',
 'DF_2303121152302944/O2collision_001;1': 'TTree',
 'DF_2303121152302944/O2filtertrack;1': 'TTree',
 'DF_2303121152302944/O2filtertrackextr;1': 'TTree',
 'DF_2303121152302944/O2filtertrackmc;1': 'TTree',
 'DF_2303121152302944/O2genparticles;1': 'TTree',
 'DF_2303121152302976;1': 'TDirectory',
 'DF_2303121152302976/O2mccollision;1': 'TTree',
 'DF_2303121152302976/O2collision_001;1': 'TTree',
 'DF_2303121152302976/O2filtertrack;1': 'TTree',
 'DF_2303121152302976/O2filtertrackextr;1': 'TTree',
 'DF_2303121152302976/O2filtertrackmc;1': 'TTree',
 'DF_2303121152302976/O2genparticles;1': 'TTree',
 'DF_2303121152303008;1': 'TDirectory',
 'DF_2303121152303008/O2mccollision;1': 'TTree',
 'DF_2303121152303008/O2collision_001;1': 'TTree',
 'DF_2303121152303008/O2filtertrack;1': 'TTree',
 'DF_2303121152303008/O2filtertrackextr;1': 'TTree',
 'DF_2303121152303008/O2filtertrackmc;1': 'TTree',
 'DF_2303121152303008

In [5]:
file["DF_2303121152302944/O2filtertrackmc"].show()

name                 | typename                 | interpretation                
---------------------+--------------------------+-------------------------------
fPdgCode             | int32_t                  | AsDtype('>i4')
fIsPhysicalPrimary   | bool                     | AsDtype('bool')
fMainHfMotherPdgCode | int32_t                  | AsDtype('>i4')
fMainBeautyAncest... | int32_t                  | AsDtype('>i4')
fMainMotherOrigIndex | int32_t                  | AsDtype('>i4')
fMainMotherNfinal... | int32_t                  | AsDtype('>i4')
fMainMotherPt        | float                    | AsDtype('>f4')
fMainMotherY         | float                    | AsDtype('>f4')
fMainBeautyAncest... | float                    | AsDtype('>f4')
fMainBeautyAncestorY | float                    | AsDtype('>f4')


## Useful variable for the MC analysis
- ### O2filtertrackmc
   One entry per reconstructed track containing simulation information of the particle associated to the reconstructed track
  - **fPdgCode**: dentifies particle species according to convention reported in PDG; e.g. π +(-) : (-)211; K +(-) :(-)321; p (p) : (-)2212;
  - **fMainHfMotherPdgCode**: pdg code of the mother when relevant, 0 otherwise
    - HF particles we want to study: $D^0 (D^0): (-)421; \ Λ_c^+ (Λ_c^-): (-)4122$;
    - $K_0^s: \ 310$;
  - **fMainMotherOrigIndex**: index of mother particle in original simulation tree.
    In simulation many particles (quarks, gluons, hadrons) are produced. Some particles may even appear
    more than once (e.g. think to a quark radiated a gluon, or to a particle which interacts with the material).
    Book keeping them all, makes the tree size very large. I saved you only the index and a filtered tree
    (O2genparticles), which contains the information of only the particles we are interested in;
  - **fMainMotherNfinalStateDaught**: number of final-state daughters, negative in
    case the final state does not match one in which we are interested (e.g. $D_0 → K^- K^+ , \ Λ_c^+ → p(K_0^s →)π^- π^+ $)
    Usage: it’s not enough that two or three particles come from the same $D_0$ or $Λ_c^+$ to identify signal, you must be sure that these $D_0$ and $Λ_c^+$ decayed in the right channel;
  - **fMainMotherPt**, **fMainMotherY**: $p_T$ and rapidity $y$ of mother particle;
  - **fMainBeautyAncestorPdgCode**: pdg code of beauty particles for cases in which the charm hadrons derive from beauty decay, 0 otherwise;
  - **fMainBeautyAncestorPt**, **fMainBeautyAncestorY**: $p_T$ and rapidity $y$ of beauty ancestor particle;

In summary: a pair of tracks corresponds to a $D_0 → K^-π^+$ decay if:
- fPdgCode are -321 and 211
- fMainHfMotherPdgCode = 421 for both
- fMainMotherOrigIndex is the same for the two tracks
- fMainMotherNfinalStateDaught = 2 (for both)

- ### O2genparticles
  Table of filtered generated particles.
As previously mentioned, in simulation many particles (quarks, gluons, hadrons) are produced.
Some particles may even appear more than once (e.g. think to a quark radiated a gluon, or to a
particle which interacts with the material). Book keeping them all, makes the tree size very large. I saved you only a filtered tree, which contains the information of only the particles we are interested in: $D_0$ and $Λ_c^+$ (and $K_s^0$)
    - **fPdgCode**: pdg code of generated particle (same convention as in O2filtertrackmc);
    - **fMainMotherPt**, **fMainMotherY**: particle pT and rapidity. Sorry for possibly misleading name: “mother” here it is used in correspondence to “main mother” in O2filtertrackmc;
    - **fMaxEtaDaughter**: max abs(pseudorapidity, η) of daughter particles. Needed to
identify reconstructable decays, for which daughters must have |η|<0.8;
    - **fMainBeautyAncestor**: [PdgCode,Pt,Y]: same as in O2filtertrackmc;
    - **fIndexMcCollisions**: match to generated collision index of tree entry in O2mccollision table → needed only to select collision with |fPosz| <10 cm;


- ### O2mccollision
   Table of generated collisions
    - **fPosX, fPosY, fPosZ**: global coordinates of collision point. (n.b.: primary vertex in O2collision_001 table gives coordinates of primary vertex, i.e. of the reconstructed position of what is assumed to be a collision)
You will need only the fPosZ variable to reject collisions generated with at
|fPosZ| <10 cm
Apply same condition in reconstruction: i.e. reject tracks from collision whose
primary vertex has |fPosZ| <10 cm

In [26]:
# visualize the tables to work with
file["DF_2303121152302944/O2collision_001"].arrays(library="pd", entry_stop=15)

,fIndexBCs,fPosX,fPosY,fPosZ,fCovXX,fCovXY,fCovYY,fCovXZ,fCovYZ,fCovZZ,fFlags,fChi2,fNumContrib,fCollisionTime,fCollisionTimeRes
0,3,-0.038224,-0.022711,4.264091,8.149073e-07,-1.326043e-09,8.032657e-07,1.405133e-07,-1.514563e-07,1.076609e-06,0,64.500000,41,0.074566,3.714844
1,3,-0.033217,-0.026897,-8.138306,2.028421e-06,-4.596077e-07,1.972541e-06,3.341120e-07,-3.504101e-07,1.822598e-06,0,236.625000,34,-1.241503,3.707031
2,34,-0.035598,-0.018980,3.706661,2.828892e-07,5.635229e-09,3.185123e-07,2.323941e-08,-3.061359e-09,3.501773e-07,0,144.750000,82,-2.045757,3.179688
3,47,-0.041739,-0.009751,-5.725029,4.020985e-07,-3.728201e-08,4.244503e-07,2.600427e-08,6.337359e-09,3.743917e-07,0,87.000000,69,-0.058566,2.158203
4,67,-0.036955,-0.028970,-0.655678,1.091510e-06,6.222399e-08,1.123175e-06,4.519825e-08,8.632196e-08,9.844080e-07,0,45.718750,37,0.166907,5.855469
5,76,-0.035140,-0.025776,2.543926,4.447997e-06,7.502967e-08,3.889203e-06,1.166947e-06,2.786983e-07,6.083399e-06,0,11.054688,11,-0.171014,7.164062
6,85,-0.032369,-0.023349,8.278152,4.664063e-06,1.994194e-07,3.986061e-06,5.378388e-07,6.291084e-07,4.250556e-06,0,10.476562,9,-0.298341,10.203125
7,93,-0.036596,-0.032305,-3.103340,4.514586e-07,5.638867e-09,5.746260e-07,1.538137e-08,7.596100e-08,4.912727e-07,0,92.437500,65,-1.382065,3.419922
8,102,-0.032120,-0.025266,-3.516178,3.522728e-07,-3.621608e-09,4.158355e-07,-9.538780e-09,-7.748895e-09,3.424939e-07,0,126.875000,56,-0.649206,3.201172
9,111,-0.047818,-0.027607,-1.038109,2.477318e-06,-6.933697e-07,2.656132e-06,-4.612375e-07,7.716008e-07,2.566725e-06,0,42.031250,19,0.002681,3.708984


In [41]:
file["DF_2303121152302944/O2filtertrackmc"].arrays(library="pd", entry_stop=15)

,fPdgCode,fIsPhysicalPrimary,fMainHfMotherPdgCode,fMainBeautyAncestorPdgCode,fMainMotherOrigIndex,fMainMotherNfinalStateDaught,fMainMotherPt,fMainMotherY,fMainBeautyAncestorPt,fMainBeautyAncestorY
0,211,True,0,0,-1,0,0.0,0.0,0.0,0.0
1,-211,True,0,0,-1,0,0.0,0.0,0.0,0.0
2,-211,True,0,0,-1,0,0.0,0.0,0.0,0.0
3,211,True,0,0,-1,0,0.0,0.0,0.0,0.0
4,211,True,0,0,-1,0,0.0,0.0,0.0,0.0
5,-211,True,0,0,-1,0,0.0,0.0,0.0,0.0
6,211,True,0,0,-1,0,0.0,0.0,0.0,0.0
7,-211,True,0,0,-1,0,0.0,0.0,0.0,0.0
8,211,True,0,0,-1,0,0.0,0.0,0.0,0.0
9,211,True,0,0,-1,0,0.0,0.0,0.0,0.0


In [40]:
z_cut_test = file["DF_2303121152302944/O2filtertrack"].arrays(library="pd")
z_cut_test = z_cut_test[z_cut_test["fZ"].abs() < 10]
row_indexis = z_cut_test.index.tolist()
print(len(z_cut_test), max(row_indexis))
z_cut_test.head(10)


KeyError: 'fZ'

In [10]:
# Filter the data to select the "good tracks"
# Perform the cut on fPosZ in the "track" file and then perform a "JOIN"
tracks_df = file["DF_2303121152302944/O2filtertrackmc"].arrays(library="pd")
tracks_df = tracks_df.loc[row_indexis]

# Select the right particles (fPdgCode)
filtered_tracks = tracks_df[tracks_df["fPdgCode"].isin([-321,211])]

# Select all the rows with fMainHfMotherPdgCode = 421 AND fMainMotherNfinalStateDaught = 2
filtered_tracks = filtered_tracks[(filtered_tracks["fMainHfMotherPdgCode"]==421) 
                    & (filtered_tracks['fMainMotherNfinalStateDaught'] == 2)]

# fMainMotherOrigIndex is the same for the grouped tracks
filtered_tracks = filtered_tracks.groupby('fMainMotherOrigIndex')

# Function to filter out the "good" couples of tracks
def check_group(group):
    part_pdg = set(group['fPdgCode'])
    return -321 in part_pdg and 211 in part_pdg and len(group) == 2 

# Applica il filtro sui gruppi
D_0_decays = filtered_tracks.filter(check_group)

# Selezionare solo decadimenti diretti
D_0_decays = D_0_decays[D_0_decays["fMainBeautyAncestorPdgCode"] == 0]

print("In this file there are", len(D_0_decays)/2, "D_0 decays")
D_0_decays.head(10)



In this file there are 10.0 D_0 decays


,fPdgCode,fIsPhysicalPrimary,fMainHfMotherPdgCode,fMainBeautyAncestorPdgCode,fMainMotherOrigIndex,fMainMotherNfinalStateDaught,fMainMotherPt,fMainMotherY,fMainBeautyAncestorPt,fMainBeautyAncestorY
535,211,True,421,0,115539,2,4.748023,0.561990,0.0,0.0
536,-321,True,421,0,115539,2,4.748023,0.561990,0.0,0.0
1278,-321,True,421,0,278327,2,2.138527,-0.015906,0.0,0.0
1284,211,True,421,0,278327,2,2.138527,-0.015906,0.0,0.0
3723,211,True,421,0,814201,2,0.323620,0.481701,0.0,0.0
3726,-321,True,421,0,814201,2,0.323620,0.481701,0.0,0.0
3968,211,True,421,0,866361,2,2.167162,0.347608,0.0,0.0
3973,-321,True,421,0,866361,2,2.167162,0.347608,0.0,0.0
3981,211,True,421,0,870070,2,4.593236,0.317109,0.0,0.0
3982,-321,True,421,0,870070,2,4.593236,0.317109,0.0,0.0


In [10]:
# Function to filter out the "good" couples of tracks
def check_group(group):
    part_pdg = set(group['fPdgCode'])
    return -321 in part_pdg and 211 in part_pdg and len(group) == 2 


In [18]:
# file by file...
all_files = file.keys(filter_name = r"DF_*")
file_ID = list(set([code.split("/")[0] for code in all_files]))
# remove the directoryes code
file_ID = [s for s in file_ID if not s.endswith(";1")]

#good_gen_part = pd.DataFrame()
#tot_gen_part = pd.DataFrame()
n_D_0 = 0

for directory in file_ID:

    tracks_df = file[directory + "/O2filtertrack"].arrays(["fIndexCollisions", ]
                                                          ,library="pd")
    tracks_mc_df = file[directory + "/O2filtertrackmc"].arrays(["fPdgCode","fMainMotherOrigIndex","fMainHfMotherPdgCode","fMainMotherNfinalStateDaught","fMainBeautyAncestorPdgCode"],library="pd")
    tracks_df = pd.merge(left=tracks_df, 
                         right=tracks_mc_df, 
                         how='inner', left_index=True, right_index=True)

    tracks_extra = file[directory + "/O2filtertrackextr"].arrays(["fEta"],library="pd")
    tracks_df = pd.merge(left=tracks_df, 
                         right=tracks_extra, 
                         how='inner', left_index=True, right_index=True)

    collision_fPosZ = file[directory + "/O2collision_001"].arrays( ["fPosZ"], library="pd")
    tracks_df = pd.merge(left=tracks_df, right=collision_fPosZ, how='inner', left_on='fIndexCollisions', right_index=True)

    # fPosZ cut
    tracks_df = tracks_df[tracks_df["fPosZ"].abs() < 10]

    # Select the right particles (fPdgCode)
    tracks_df = tracks_df[tracks_df["fPdgCode"].isin([-321,211])]
    
    # Select all the rows with fMainHfMotherPdgCode = 421 AND fMainMotherNfinalStateDaught = 2
    tracks_df = tracks_df[(tracks_df["fMainHfMotherPdgCode"] == 421) 
                        & (tracks_df['fMainMotherNfinalStateDaught'] == 2)]

    # Pseudorapidity cut
    tracks_df = tracks_df[tracks_df["fEta"].abs() < 0.8]

    # Check on the fMainBeautyAncestor = 0
    tracks_df = tracks_df[tracks_df["fMainBeautyAncestorPdgCode"] == 0]

    
    # fMainMotherOrigIndex is the same for the grouped tracks
    tracks_df = tracks_df.groupby('fMainMotherOrigIndex')
   

    

    #defined pT interval in which you can/want to “make the measurement”
    # For now no selection on pT...TO DO LATER
    
    # Applica il filtro sui gruppi
    D_0_decays = tracks_df.filter(check_group)


    # Contiamo le D_0 e sommiamole per avere un totale
    n_D_0 += len(D_0_decays)/2

    print("In this file there are", len(D_0_decays)/2, "D_0 decays")
    
    

In this file there are 8.0 D_0 decays
In this file there are 4.0 D_0 decays
In this file there are 2.0 D_0 decays
In this file there are 5.0 D_0 decays
In this file there are 6.0 D_0 decays
In this file there are 12.0 D_0 decays
In this file there are 3.0 D_0 decays
In this file there are 6.0 D_0 decays
In this file there are 6.0 D_0 decays
In this file there are 5.0 D_0 decays
In this file there are 10.0 D_0 decays
In this file there are 0.0 D_0 decays
In this file there are 3.0 D_0 decays
In this file there are 5.0 D_0 decays
In this file there are 8.0 D_0 decays
In this file there are 7.0 D_0 decays
In this file there are 6.0 D_0 decays
In this file there are 3.0 D_0 decays
In this file there are 6.0 D_0 decays
In this file there are 6.0 D_0 decays
In this file there are 4.0 D_0 decays
In this file there are 9.0 D_0 decays
In this file there are 8.0 D_0 decays
In this file there are 1.0 D_0 decays
In this file there are 8.0 D_0 decays
In this file there are 9.0 D_0 decays
In this fi

In [20]:
n_D_0

1271.0

In [ ]:
# Do the same of the previous cell but with all the data (not only one file)
# Prepare the data
file_names_mc = file.keys(filter_name=r"*O2filtertrackmc")
file_names = file.keys(filter_name=r"*O2filtertrack")
all_tracks_mc = pd.concat( [file[name_string].arrays(library="pd") for name_string in file_names_mc] )
all_tracks = pd.concat( [file[name_string].arrays(library="pd") for name_string in file_names] )

# Select the rows to mantain
z_cut = all_tracks[all_tracks["fZ"].abs() < 10]
row_indexis = z_cut.index.tolist() 

# Perform the cut on the z coordinate
all_tracks_mc = all_tracks_mc.loc[row_indexis]

# Filter the data selecting the right particles that mark the D_0 decay
filtered_tracks = all_tracks_mc[all_tracks_df["fPdgCode"].isin([-321,211])]

# Select all the rows with fMainHfMotherPdgCode = 421 AND fMainMotherNfinalStateDaught = 2
filtered_tracks = filtered_tracks[(filtered_tracks["fMainHfMotherPdgCode"]==421) 
                    & (filtered_tracks['fMainMotherNfinalStateDaught'] == 2)]

# fMainMotherOrigIndex is the same for the grouped tracks
filtered_tracks = filtered_tracks.groupby('fMainMotherOrigIndex')

# Function to filter out the "good" couples of tracks
def check_group(group):
    part_pdg = set(group['fPdgCode'])
    return -321 in part_pdg and 211 in part_pdg and len(group) == 2 

# Applica il filtro sui gruppi
D_0_decays = filtered_tracks.filter(check_group)

print("In the full dataset there are", len(D_0_decays)/2, "D_0 decays")
#D_0_decays.head(10)

In [21]:
# Now we have to check if these tracks come from collisions with |fPosZ| < 10 cm
# Start checking this in the O2mccollision tables
# Select the collisions with fPosZ < 10 cm
#file_names_collisions = file.keys(filter_name=r"*O2mccollision")
#mccollision_df = pd.concat( [ file[name].arrays(library="pd") for name in file_names_collisions ] )

mccollision_df = file["DF_2303121152302944/O2mccollision"].arrays(library="pd")
mask_mccollision_pZ_cut = mccollision_df["fPosZ"].abs() < 10

# visualize the result
#print(len(mccollision_pZ_cut), len(mccollision_df))
#mccollision_pZ_cut.head(10)
print(mask_mccollision_pZ_cut)
type(mask_mccollision_pZ_cut)

0      True
1      True
2      True
3      True
4      True
       ... 
938    True
939    True
940    True
941    True
942    True
Name: fPosZ, Length: 943, dtype: bool


pandas.core.series.Series

In [23]:
# Import all the data in a single dataframe
#file_names_gen_par = file.keys(filter_name=r"*O2genparticles")
#generated_part = pd.concat( [ file[name].arrays(library="pd") for name in file_names_gen_par ] )

# Match the ID of the two tables
generated_part = file["DF_2303121152302944/O2genparticles"].arrays(library="pd")
#good_generated_particles = generated_part[generated_part['fIndexMcCollisions'].isin(mccollision_pZ_cut['fIndexBCs'])]

good_generated_particles = pd.merge(left=generated_part, right=mccollision_df['fPosZ'], how='inner', left_on='fIndexMcCollisions', right_index=True)
good_generated_particles = good_generated_particles[good_generated_particles["fPosZ"].abs() < 10]

print(len(good_generated_particles), len(generated_part))
good_generated_particles.head(10)

2782 3032


,fPdgCode,fIndexMcCollisions,fMainBeautyAncestorPdgCode,fMainMotherPt,fMainMotherY,fMaxEtaDaughter,fMainBeautyAncestorPt,fMainBeautyAncestorY,fPosZ
0,-421,0,0,0.598802,1.470003,2.749113,0.0,0.0,-2.089081
1,310,0,0,0.459426,0.168955,0.961717,0.0,0.0,-2.089081
2,421,0,0,3.816828,-0.358111,0.508804,0.0,0.0,-2.089081
3,310,1,0,0.378431,-4.804149,5.214464,0.0,0.0,4.264450
4,310,2,0,0.350817,-3.212667,3.566491,0.0,0.0,-8.141159
5,310,2,0,0.651646,2.237123,2.560400,0.0,0.0,-8.141159
6,421,4,0,3.626508,0.730199,0.907534,0.0,0.0,3.706577
7,310,4,0,0.378373,2.147589,2.526597,0.0,0.0,3.706577
8,310,4,0,1.208306,-1.136796,1.254971,0.0,0.0,3.706577
9,-4122,4,0,1.508752,3.929880,4.216262,0.0,0.0,3.706577


In [21]:
# file by file...
all_files = file.keys(filter_name = r"DF_*")
file_ID = list(set([code.split("/")[0] for code in all_files]))
# remove the directoryes code
file_ID = [s for s in file_ID if not s.endswith(";1")]

good_gen_part = pd.DataFrame()
tot_gen_part = pd.DataFrame()

for directory in file_ID:

     # "JOIN"
    generated_part = file[directory + "/O2genparticles"].arrays(library="pd")
    good_generated_particles = generated_part[generated_part['fIndexMcCollisions'].isin(mccollision_pZ_cut['fIndexBCs'])]

    
    # fPosZ cut
    mccollision_df = file[directory + "/O2mccollision"].arrays(library="pd")
    mccollision_pZ_cut = mccollision_df[mccollision_df["fPosZ"].abs() < 10]

    # Pseudorapidity cut

    # Check on the fMainBeautyAncestor = 0

    #defined pT interval in which you can/want to “make the measurement”
    # For now no selection on pT...TO DO LATER 
    
    #results
    good_gen_part = pd.concat([good_gen_part, good_generated_particles], ignore_index=True) 
    tot_gen_part = pd.concat([tot_gen_part, generated_part], ignore_index=True) 

print("The generated particles before check the Z coordinate are:", len(tot_gen_part))
print("After the cut on Z_collision < 10 cm we have ", len(good_gen_part), "particles")

NameError: name 'mccollision_pZ_cut' is not defined

In [15]:
good_gen_part.head(3)

,fPdgCode,fIndexMcCollisions,fMainBeautyAncestorPdgCode,fMainMotherPt,fMainMotherY,fMaxEtaDaughter,fMainBeautyAncestorPt,fMainBeautyAncestorY
0,4122,0,0,4.953534,1.226256,1.399188,0.0,0.0
1,-4122,0,0,3.655818,1.312502,1.735824,0.0,0.0
2,310,0,0,0.584869,1.535978,1.632219,0.0,0.0


In [47]:
# Select all the rows with fPdgCode = 421 
good_gen_part = good_gen_part[(good_gen_part["fPdgCode"]==421)]

#defined pT interval in which you can/want to “make the measurement”
# For now no selection on pT...TO DO LATER

# Pseudorapidity cut
good_gen_part = good_gen_part[good_gen_part["fMainMotherY"].abs() < 0.8]

# Check on the fMainBeautyAncestor = 0
good_gen_part = good_gen_part[good_gen_part["fMainBeautyAncestorY"].abs() == 0]
#------------------------------------------------------efficenza per segnale generato = raw yeld
#Visualize the results --> efficiency denominator
print("In the MC simulation have been generated", len(good_gen_part) ," D_0 mesons")

In the MC simulation have been generated 3067  D_0 mesons


In [17]:
n_tot_collisions = len(mccollision_pZ_cut_df)
print("Total number of collisions: ", n_tot_collisions)
n_fposZ = len(mccollision_pZ_cut)
print("Number of collisions with |fPosZ| < 10 cm: ", n_fposZ)

NameError: name 'mccollision_pZ_cut_df' is not defined

In [18]:
# Select the particle from the O2filtertrackmc table
tracks_df = file["DF_2303121152302944/O2filtertrackmc"].arrays(library="pd")
selected_tracks = tracks_df[tracks_df["fMainMotherY"].abs() < 0.8]
selected_tracks.head()

,fPdgCode,fIsPhysicalPrimary,fMainHfMotherPdgCode,fMainBeautyAncestorPdgCode,fMainMotherOrigIndex,fMainMotherNfinalStateDaught,fMainMotherPt,fMainMotherY,fMainBeautyAncestorPt,fMainBeautyAncestorY
0,211,True,0,0,-1,0,0.0,0.0,0.0,0.0
1,-211,True,0,0,-1,0,0.0,0.0,0.0,0.0
2,-211,True,0,0,-1,0,0.0,0.0,0.0,0.0
3,211,True,0,0,-1,0,0.0,0.0,0.0,0.0
4,211,True,0,0,-1,0,0.0,0.0,0.0,0.0


In [50]:
efficiency = n_D_0/len(good_gen_part)
print(efficiency)

0.4144114770133681
